In [1]:
import torch
import transformers

In [2]:
model = transformers.AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")

In [3]:
for parent in model.__class__.__mro__:
    print(parent)

<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
<class 'transformers.models.llama.modeling_llama.LlamaPreTrainedModel'>
<class 'transformers.modeling_utils.PreTrainedModel'>
<class 'torch.nn.modules.module.Module'>
<class 'transformers.modeling_utils.ModuleUtilsMixin'>
<class 'transformers.utils.hub.PushToHubMixin'>
<class 'transformers.integrations.peft.PeftAdapterMixin'>
<class 'transformers.generation.utils.GenerationMixin'>
<class 'transformers.generation.continuous_batching.ContinuousMixin'>
<class 'object'>


In [4]:
for name, child in model.named_children():
    print(name, type(child))

model <class 'transformers.models.llama.modeling_llama.LlamaModel'>
lm_head <class 'torch.nn.modules.linear.Linear'>


In [5]:
def traverse(module, indent=0):
    for name, child in module.named_children():
        print("\t" * indent, name, type(child))
        traverse(child, indent + 1)


traverse(model)

 model <class 'transformers.models.llama.modeling_llama.LlamaModel'>
	 embed_tokens <class 'torch.nn.modules.sparse.Embedding'>
	 layers <class 'torch.nn.modules.container.ModuleList'>
		 0 <class 'transformers.models.llama.modeling_llama.LlamaDecoderLayer'>
			 self_attn <class 'transformers.models.llama.modeling_llama.LlamaAttention'>
				 q_proj <class 'torch.nn.modules.linear.Linear'>
				 k_proj <class 'torch.nn.modules.linear.Linear'>
				 v_proj <class 'torch.nn.modules.linear.Linear'>
				 o_proj <class 'torch.nn.modules.linear.Linear'>
			 mlp <class 'transformers.models.llama.modeling_llama.LlamaMLP'>
				 gate_proj <class 'torch.nn.modules.linear.Linear'>
				 up_proj <class 'torch.nn.modules.linear.Linear'>
				 down_proj <class 'torch.nn.modules.linear.Linear'>
				 act_fn <class 'torch.nn.modules.activation.SiLU'>
			 input_layernorm <class 'transformers.models.llama.modeling_llama.LlamaRMSNorm'>
			 post_attention_layernorm <class 'transformers.models.llama.modeling_llam

In [6]:
tokenizer = transformers.AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")

In [7]:
encodings = tokenizer("Hello!", return_tensors="pt")
print(encodings)

with torch.no_grad():
    logits = model(**encodings)
print(logits)

{'input_ids': tensor([[19556,    17]]), 'attention_mask': tensor([[1, 1]])}
CausalLMOutputWithPast(loss=None, logits=tensor([[[ 18.6710,   3.8383,   3.9483,  ...,  13.2351,  14.9483,  10.6846],
         [  8.5299, -12.0491, -12.0441,  ...,  -2.5762,   5.6361,  -2.8067]]]), past_key_values=<transformers.cache_utils.DynamicCache object at 0x15367b680>, hidden_states=None, attentions=None)


In [ ]:
import torch.fx as fx
import torch.nn as nn
import torch.ops

In [ ]:
class Model(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.register_module("activation", nn.ReLU())
        self.register_buffer("b", torch.Tensor([1.0]))

    def forward(self, a: torch.Tensor) -> torch.Tensor:
        c = a + self.b
        d = self.activation(c)
        return d


model = Model()

graph = fx.Graph()
node_a = graph.placeholder("a")
node_b = graph.get_attr("b")
node_add = graph.call_function(torch.ops.aten.add.Tensor, args=(node_a, node_b))
node_relu = graph.call_module("activation", args=(node_add,))
_ = graph.output(node_relu)

graph_module = fx.GraphModule(root=model, graph=graph)

In [10]:
import torch

a = torch.Tensor([-2.0])
d = model(a)
print(d)

tensor([0.])


In [11]:
graph.print_tabular()

opcode         name        target      args           kwargs
-------------  ----------  ----------  -------------  --------
placeholder    a           a           ()             {}
get_attr       b           b           ()             {}
call_function  add         aten.add    (a, b)         {}
call_module    activation  activation  (add,)         {}
output         output      output      (activation,)  {}


In [12]:
graph_module.graph.print_tabular()

opcode         name        target      args           kwargs
-------------  ----------  ----------  -------------  --------
placeholder    a           a           ()             {}
get_attr       b           b           ()             {}
call_function  add         aten.add    (a, b)         {}
call_module    activation  activation  (add,)         {}
output         output      output      (activation,)  {}


In [ ]:
import torch.export

In [ ]:
graph_module = torch.export.export(model, (a,))
graph_module.graph.print_tabular()

opcode         name    target             args        kwargs
-------------  ------  -----------------  ----------  --------
placeholder    c_b     c_b                ()          {}
placeholder    a       a                  ()          {}
call_function  add     aten.add.Tensor    (a, c_b)    {}
call_function  relu    aten.relu.default  (add,)      {}
output         output  output             ((relu,),)  {}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
